# 문제 18. 코호트별 LTV 곡선과 획득비 상한

`common_order_items.py`를 import하지 않고, 그 안에 있던 정제 로직까지 전부
이 노트북 안에서 단계별 셀로 직접 풀어냅니다. 위에서부터 순서대로 실행하면서
각 단계의 중간 데이터프레임을 눈으로 확인하며 따라갈 수 있습니다.

**전제**: 이 노트북은 `data/` 폴더(orders.csv, order_items.csv, products.csv)와
같은 위치에서 실행합니다.

## 0. 환경 설정

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 50)

# 데이터 기간(2024 상반기)의 마지막 날 = 관측 종료일
OBS_END = pd.Timestamp("2024-06-30")


---
# PART A. 표준 순매출 원장 만들기

(1장 문제01 로직을 그대로 이 노트북 안에서 직접 재현합니다)

## 1. 원본 데이터 적재

In [2]:
orders = pd.read_csv("./data/orders.csv")
items = pd.read_csv("./data/order_items.csv")

print("orders shape:", orders.shape)
print("items shape:", items.shape)
orders.head()


orders shape: (200000, 5)
items shape: (500000, 6)


,order_id,customer_id,order_datetime,channel,status
0,25463,1077,2024-06-25 00:17:41,store,delivered
1,171088,2998,2024-05-28 19:35:20,web,delivered
2,27437,4066,2024-02-14 17:49:14,app,delivered
3,98425,3243,2024-05-08 16:37:36,store,delivered
4,22325,5331,2024-04-17 20:08:22,app,delivered


## 2. 중복 제거

In [3]:
n_orders_before, n_items_before = len(orders), len(items)

orders = orders.drop_duplicates()
items = items.drop_duplicates()

print(f"orders 중복 제거: {n_orders_before} -> {len(orders)} ({n_orders_before - len(orders)}건 손실)")
print(f"items  중복 제거: {n_items_before} -> {len(items)} ({n_items_before - len(items)}건 손실)")


orders 중복 제거: 200000 -> 200000 (0건 손실)
items  중복 제거: 500000 -> 499880 (120건 손실)


## 3. 날짜 파싱 (`errors="coerce"`)

변환 실패(빈 문자열 등)는 에러 대신 `NaT`로 채워집니다.

In [4]:
orders["order_datetime"] = pd.to_datetime(orders["order_datetime"], errors="coerce")

n_nat = orders["order_datetime"].isna().sum()
print(f"날짜 파싱 실패(NaT) 건수: {n_nat}")


날짜 파싱 실패(NaT) 건수: 1933


## 4. 기간 필터 (2024년 상반기: 1/1 ~ 6/30)

In [5]:
period_mask = (orders["order_datetime"] >= "2024-01-01") & (orders["order_datetime"] < "2024-07-01")
orders_in_period = orders[period_mask].copy()

print(f"기간 필터 전: {len(orders):,}건")
print(f"기간 필터 후: {len(orders_in_period):,}건 (제외 {len(orders) - len(orders_in_period):,}건, NaT 포함)")


기간 필터 전: 200,000건
기간 필터 후: 198,027건 (제외 1,973건, NaT 포함)


## 5. orders x order_items 결합 (inner join)

In [6]:
order_items = orders_in_period.merge(items, on="order_id", how="inner")

print(f"결합 후 order_items: {len(order_items):,}건")
order_items.head()


결합 후 order_items: 494,994건


,order_id,customer_id,order_datetime,channel,status,order_item_id,product_id,quantity,unit_price,discount
0,25463,1077,2024-06-25 00:17:41,store,delivered,265800,104,5,"73,300.00",0.40
1,25463,1077,2024-06-25 00:17:41,store,delivered,16468,319,4,"68,800.00",0.21
2,25463,1077,2024-06-25 00:17:41,store,delivered,477391,9,4,"19,400.00",0.34
3,171088,2998,2024-05-28 19:35:20,web,delivered,55193,279,4,"28,000.00",0.02
4,171088,2998,2024-05-28 19:35:20,web,delivered,200065,3,2,"20,300.00",0.32


## 6. `unit_price` 결측 제외 + `line_amount` 파생

`line_amount = quantity x unit_price x (1 - discount)`

In [7]:
n_before = len(order_items)
# unit_price null인건 drop 처리
order_items = order_items.dropna(subset=["unit_price"])
print(f"unit_price 결측 제외: {n_before:,} -> {len(order_items):,}건")

# line_amount 매출금액 계산
order_items["line_amount"] = (
    order_items["quantity"] * order_items["unit_price"] * (1 - order_items["discount"])
)


unit_price 결측 제외: 494,994 -> 480,025건


## 7. 정상건(`is_net`) 플래그 및 순매출 라인 추출

`canceled`, `returned` 상태를 제외한 나머지가 순매출(net) 라인입니다.

In [8]:
# 순매출건만 추출
order_items["is_net"] = ~order_items["status"].isin(["canceled", "returned"])

net_lines = order_items[order_items["is_net"]].copy()
print(f"전체 라인: {len(order_items):,}건 / 순매출 라인: {len(net_lines):,}건")
print(f"고유 고객 수: {net_lines['customer_id'].nunique():,}명")
net_lines.head()


전체 라인: 480,025건 / 순매출 라인: 407,686건
고유 고객 수: 5,714명


,order_id,customer_id,order_datetime,channel,status,order_item_id,product_id,quantity,unit_price,discount,line_amount,is_net
0,25463,1077,2024-06-25 00:17:41,store,delivered,265800,104,5,"73,300.00",0.40,"219,900.00",True
1,25463,1077,2024-06-25 00:17:41,store,delivered,16468,319,4,"68,800.00",0.21,"217,408.00",True
2,25463,1077,2024-06-25 00:17:41,store,delivered,477391,9,4,"19,400.00",0.34,"51,216.00",True
3,171088,2998,2024-05-28 19:35:20,web,delivered,55193,279,4,"28,000.00",0.02,"109,760.00",True
4,171088,2998,2024-05-28 19:35:20,web,delivered,200065,3,2,"20,300.00",0.32,"27,608.00",True


---
# PART B. 상품 원가 결합 + 마진 계산

## 8. 상품 원가(cost) 결합

판매 단가는 오염된 `products.price`가 아니라 거래 시점 값인 `order_items.unit_price`를 그대로 씁니다.
원가(`cost`)만 상품 마스터에서 가져옵니다.

In [9]:
# 상품 중복건 제거
products = pd.read_csv("./data/products.csv").drop_duplicates()

# net_lines 순매출건만 추출
# 순매출과 상품 join
# net =  순매출건 join product
net = net_lines.merge(products[["product_id", "cost"]], on="product_id", how="left")

n_cost_null = net["cost"].isna().sum()
print(f"원가(cost) 결측 라인: {n_cost_null:,}건")
net[["product_id", "unit_price", "cost"]].head()


원가(cost) 결측 라인: 0건


,product_id,unit_price,cost
0,104,"73,300.00","49,200.00"
1,319,"68,800.00","40,900.00"
2,9,"19,400.00","11,000.00"
3,279,"28,000.00","15,400.00"
4,3,"20,300.00","10,500.00"


## 9. 라인별 마진(margin) 계산

`margin = quantity x (unit_price x (1 - discount) - cost)`

In [10]:
# net =  순매출건 join product
net["margin"] = net["quantity"] * (
    net["unit_price"] * (1 - net["discount"]) - net["cost"]
)

net[["customer_id", "order_datetime", "quantity", "unit_price", "discount", "cost", "line_amount", "margin"]].head()


,customer_id,order_datetime,quantity,unit_price,discount,cost,line_amount,margin
0,1077,2024-06-25 00:17:41,5,"73,300.00",0.40,"49,200.00","219,900.00","-26,100.00"
1,1077,2024-06-25 00:17:41,4,"68,800.00",0.21,"40,900.00","217,408.00","53,808.00"
2,1077,2024-06-25 00:17:41,4,"19,400.00",0.34,"11,000.00","51,216.00","7,216.00"
3,2998,2024-05-28 19:35:20,4,"28,000.00",0.02,"15,400.00","109,760.00","48,160.00"
4,2998,2024-05-28 19:35:20,2,"20,300.00",0.32,"10,500.00","27,608.00","6,608.00"


---
# PART C. 코호트별 누적 LTV 곡선

## 10. 주문월(order_month) 파생

날짜를 '연-월' 단위(Period)로 뭉갭니다. 예: 2024-03-15 -> 2024-03

In [11]:
# net =  순매출건 join product
net["order_month"] = net["order_datetime"].dt.to_period("M")

net[["customer_id", "order_datetime", "order_month"]].head()

print(net.head())


   order_id  customer_id      order_datetime channel     status  \
0     25463         1077 2024-06-25 00:17:41   store  delivered   
1     25463         1077 2024-06-25 00:17:41   store  delivered   
2     25463         1077 2024-06-25 00:17:41   store  delivered   
3    171088         2998 2024-05-28 19:35:20     web  delivered   
4    171088         2998 2024-05-28 19:35:20     web  delivered   

   order_item_id  product_id  quantity  unit_price  discount  line_amount  \
0         265800         104         5   73,300.00      0.40   219,900.00   
1          16468         319         4   68,800.00      0.21   217,408.00   
2         477391           9         4   19,400.00      0.34    51,216.00   
3          55193         279         4   28,000.00      0.02   109,760.00   
4         200065           3         2   20,300.00      0.32    27,608.00   

   is_net      cost     margin order_month  
0    True 49,200.00 -26,100.00     2024-06  
1    True 40,900.00  53,808.00     2024-06  

## 11. 고객별 첫 구매월(코호트) 계산

고객별로 가장 이른 `order_month`를 뽑아 `cohort`라는 이름으로 저장합니다.
이름을 바꾸는 이유(`rename`)는, 뒤에서 merge할 때 원본 `order_month` 열과 이름이 겹치지 않게 하기 위함입니다.

In [12]:
# net =  순매출건 join product
# first_month : 최초주문월
first_month = net.groupby("customer_id")["order_month"].min().rename("cohort")

print(first_month.info())
print('-----------------------')
print('-----------------------')
print('-----------------------')
print(first_month.head())


<class 'pandas.Series'>
Index: 5714 entries, 1000 to 989995
Series name: cohort
Non-Null Count  Dtype    
--------------  -----    
5714 non-null   period[M]
dtypes: period[M](1)
memory usage: 89.3 KB
None
-----------------------
-----------------------
-----------------------
customer_id
1000    2024-01
1001    2024-01
1002    2024-01
1003    2024-01
1004    2024-01
Name: cohort, dtype: period[M]


## 12. 코호트 정보를 원본에 결합

이제 `net`에는 `order_month`(그 라인이 실제 발생한 달)와 `cohort`(그 고객의 첫 구매월)
두 개의 서로 다른 열이 함께 존재하게 됩니다.

In [13]:
# net =  순매출건 join product
# first_month : 최초주문월
net = net.merge(first_month, on="customer_id")

net[["customer_id", "order_month", "cohort"]].drop_duplicates().sort_values("customer_id").head(10)


,customer_id,order_month,cohort
74356,1000,2024-01,2024-01
18777,1000,2024-02,2024-01
208919,1000,2024-06,2024-01
41073,1000,2024-05,2024-01
5074,1000,2024-04,2024-01
84362,1001,2024-03,2024-01
76745,1001,2024-02,2024-01
147537,1001,2024-01,2024-01
135821,1001,2024-04,2024-01
226641,1001,2024-05,2024-01


## 13. 경과월(period_n) 계산

`order_month - cohort`는 Period끼리의 뺄셈이라 개월수 offset 객체가 나옵니다.
`.apply(lambda x: x.n)`으로 그 안의 순수 정수(개월수)만 꺼냅니다.

예: 1월 코호트 고객이 3월에 산 라인 -> period_n = 2 (가입 후 2개월째)

In [14]:
# net =  순매출건 join product
# first_month : 최초주문월
# apply 개월수 뽑기
net["period_n"] = (net["order_month"] - net["cohort"]).apply(lambda x: x.n)
net[["customer_id", "cohort", "order_month", "period_n"]].drop_duplicates().sort_values(["customer_id", "period_n"]).head(10)



,customer_id,cohort,order_month,period_n
74356,1000,2024-01,2024-01,0
18777,1000,2024-01,2024-02,1
5074,1000,2024-01,2024-04,3
41073,1000,2024-01,2024-05,4
208919,1000,2024-01,2024-06,5
147537,1001,2024-01,2024-01,0
76745,1001,2024-01,2024-02,1
84362,1001,2024-01,2024-03,2
135821,1001,2024-01,2024-04,3
226641,1001,2024-01,2024-05,4


## 14. 코호트 x 경과월 단위로 매출·마진 집계

In [15]:
# net =  순매출건 join product
# first_month : 최초주문월
# apply 개월수 뽑기
# period_n : 가입후 최조주문월 대비 개월수
# monthly 월별 기간별 매출/마진 집계
monthly = net.groupby(["cohort", "period_n"]).agg(
    매출=("line_amount", "sum"),
    마진=("margin", "sum"),
).reset_index()

monthly.head(10)


,cohort,period_n,매출,마진
0,2024-01,0,"5,527,055,665.00","1,263,187,465.00"
1,2024-01,1,"5,627,141,798.00","1,291,004,998.00"
2,2024-01,2,"6,941,864,879.00","1,586,742,579.00"
3,2024-01,3,"7,510,779,808.00","1,730,714,408.00"
4,2024-01,4,"8,510,776,064.00","1,935,717,764.00"
5,2024-01,5,"9,336,475,046.00","2,153,686,846.00"
6,2024-02,0,"450,414,796.00","102,642,396.00"
7,2024-02,1,"420,965,289.00","94,362,289.00"
8,2024-02,2,"466,365,420.00","110,349,620.00"
9,2024-02,3,"527,386,506.00","120,244,406.00"


## 15. 코호트 크기로 나눠 '1인당' 값 만들기

코호트마다 인원수가 다르므로, 총액을 그대로 비교하면 불공정합니다.
코호트별 고유 고객 수(`nunique`)로 나눠 1인당 매출/마진으로 바꿉니다.

In [16]:
# net =  순매출건 join product
# first_month : 최초주문월
# apply 개월수 뽑기
# period_n : 가입후 최조주문월 대비 개월수
# monthly 월별 기간별 매출/마진 집계
# cohort_size : 코호트별 인원수
cohort_size = net.groupby("cohort")["customer_id"].nunique()
print("코호트별 인원수:")
print(cohort_size)

monthly["cohort_size"] = monthly["cohort"].map(cohort_size)
monthly["1인당매출"] = monthly["매출"] / monthly["cohort_size"]
monthly["1인당마진"] = monthly["마진"] / monthly["cohort_size"]

monthly.head(10)


코호트별 인원수:
cohort
2024-01    4030
2024-02     850
2024-03     323
2024-04     181
2024-05     160
2024-06     170
Freq: M, Name: customer_id, dtype: int64


,cohort,period_n,매출,마진,cohort_size,1인당매출,1인당마진
0,2024-01,0,"5,527,055,665.00","1,263,187,465.00",4030,"1,371,477.83","313,446.02"
1,2024-01,1,"5,627,141,798.00","1,291,004,998.00",4030,"1,396,313.10","320,348.63"
2,2024-01,2,"6,941,864,879.00","1,586,742,579.00",4030,"1,722,547.12","393,732.65"
3,2024-01,3,"7,510,779,808.00","1,730,714,408.00",4030,"1,863,717.07","429,457.67"
4,2024-01,4,"8,510,776,064.00","1,935,717,764.00",4030,"2,111,855.10","480,326.99"
5,2024-01,5,"9,336,475,046.00","2,153,686,846.00",4030,"2,316,743.19","534,413.61"
6,2024-02,0,"450,414,796.00","102,642,396.00",850,"529,899.76","120,755.76"
7,2024-02,1,"420,965,289.00","94,362,289.00",850,"495,253.28","111,014.46"
8,2024-02,2,"466,365,420.00","110,349,620.00",850,"548,665.20","129,823.08"
9,2024-02,3,"527,386,506.00","120,244,406.00",850,"620,454.71","141,464.01"


## 16. 경과월 순서로 정렬 후 누적합(cumsum)

`groupby("cohort")["1인당매출"].cumsum()`은 같은 코호트 안에서 경과월 순서대로
값을 차곡차곡 더해나갑니다. 이게 바로 '가입 후 N개월까지 누적된 1인당 가치' 곡선입니다.

In [17]:
# net =  순매출건 join product
# first_month : 최초주문월
# apply 개월수 뽑기
# period_n : 가입후 최조주문월 대비 개월수
# monthly 월별 기간별 매출/마진 집계
# cohort_size : 코호트별 인원수
monthly = monthly.sort_values(["cohort", "period_n"])
monthly["누적1인당매출"] = monthly.groupby("cohort")["1인당매출"].cumsum()
monthly["누적1인당마진"] = monthly.groupby("cohort")["1인당마진"].cumsum()

monthly[["cohort", "period_n", "1인당매출", "누적1인당매출", "1인당마진", "누적1인당마진"]].head(10)


,cohort,period_n,1인당매출,누적1인당매출,1인당마진,누적1인당마진
0,2024-01,0,"1,371,477.83","1,371,477.83","313,446.02","313,446.02"
1,2024-01,1,"1,396,313.10","2,767,790.93","320,348.63","633,794.66"
2,2024-01,2,"1,722,547.12","4,490,338.05","393,732.65","1,027,527.31"
3,2024-01,3,"1,863,717.07","6,354,055.12","429,457.67","1,456,984.98"
4,2024-01,4,"2,111,855.10","8,465,910.23","480,326.99","1,937,311.96"
5,2024-01,5,"2,316,743.19","10,782,653.41","534,413.61","2,471,725.57"
6,2024-02,0,"529,899.76","529,899.76","120,755.76","120,755.76"
7,2024-02,1,"495,253.28","1,025,153.04","111,014.46","231,770.22"
8,2024-02,2,"548,665.20","1,573,818.24","129,823.08","361,593.30"
9,2024-02,3,"620,454.71","2,194,272.95","141,464.01","503,057.31"


## 17. 코호트별 누적 1인당 마진 곡선표 (pivot)

행은 코호트(가입월), 열은 경과월로 펼쳐서 곡선을 한눈에 보는 표를 만듭니다.

In [18]:
# net =  순매출건 join product
# first_month : 최초주문월
# apply 개월수 뽑기
# period_n : 가입후 최조주문월 대비 개월수
# monthly 월별 기간별 매출/마진 집계
# cohort_size : 코호트별 인원수
curve = monthly.pivot(index="cohort", columns="period_n", values="누적1인당마진")
curve.round(0)


period_n,0,1,2,3,4,5
cohort,,,,,,
2024-01,"313,446.00","633,795.00","1,027,527.00","1,456,985.00","1,937,312.00","2,471,726.00"
2024-02,"120,756.00","231,770.00","361,593.00","503,057.00","646,374.00",NaN
2024-03,"101,448.00","184,099.00","261,254.00","357,796.00",NaN,NaN
2024-04,"79,918.00","123,074.00","163,050.00",NaN,NaN,NaN
2024-05,"64,155.00","66,191.00",NaN,NaN,NaN,NaN
2024-06,"70,642.00",NaN,NaN,NaN,NaN,NaN


### (참고) 곡선 형태 읽는 법

- 경과월 0 -> 1 구간의 기울기가 가파르면: **초기 집중형** (가입 직후 몰아서 씀 -> 획득비 회수가 빠름)
- 완만하게 계속 우상향하면: **지속 누적형** (꾸준히 씀 -> 회수에 시간이 더 걸림)

위 표에서 각 행(코호트)이 경과월이 늘어남에 따라 얼마나 가파르게 누적되는지 직접 비교해보세요.

---
# PART D. 첫 구매 후 90일 시점 마진

## 18. 고객별 첫 구매일(정확한 날짜) 계산

지금까지의 곡선은 '월' 단위 근사치였습니다. 90일 마진은 더 정밀하게,
'첫 구매일로부터 실제 날짜 90일'을 기준으로 다시 계산합니다.

In [19]:
# net =  순매출건 join product
# first_month : 최초주문월
# apply 개월수 뽑기
# period_n : 가입후 최조주문월 대비 개월수
# monthly 월별 기간별 매출/마진 집계
# cohort_size : 코호트별 인원수
first_date = net.groupby("customer_id")["order_datetime"].min()
first_date.head()


customer_id
1000   2024-01-30 08:13:14
1001   2024-01-18 17:43:25
1002   2024-01-06 10:37:15
1003   2024-01-03 09:42:56
1004   2024-01-01 04:23:36
Name: order_datetime, dtype: datetime64[us]

## 19. 관측이 완료된 고객만 선별 (검열 처리)

우리 데이터는 2024-06-30까지만 있습니다. 만약 어떤 고객의 '첫 구매일+90일'이
6/30을 넘어선다면, 그 90일 구간이 아직 다 관측되지 않은 것이므로 평균에서 제외해야 합니다.

In [20]:
# net =  순매출건 join product
# first_month : 최초주문월
# apply 개월수 뽑기
# period_n : 가입후 최조주문월 대비 개월수
# monthly 월별 기간별 매출/마진 집계
# cohort_size : 코호트별 인원수
# first_date : 최초주문일자
print(first_date)
cutoff_ok = (first_date + pd.Timedelta(days=90)) <= OBS_END
# print('cutoff_ok : \n' , cutoff_ok)
eligible_customers = cutoff_ok[cutoff_ok].index
print('eligible_customers : ' , eligible_customers)

print(f"전체 고객 수: {len(first_date):,}명")
print(f"관측 완료(90일 창이 6/30 이내인) 고객 수: {len(eligible_customers):,}명")
print(f"제외된(아직 90일이 안 지난) 고객 수: {len(first_date) - len(eligible_customers):,}명")


customer_id
1000     2024-01-30 08:13:14
1001     2024-01-18 17:43:25
1002     2024-01-06 10:37:15
1003     2024-01-03 09:42:56
1004     2024-01-01 04:23:36
                 ...        
989712   2024-05-20 14:24:49
989772   2024-01-10 04:09:52
989922   2024-06-29 03:27:02
989977   2024-01-14 00:14:02
989995   2024-05-27 01:54:32
Name: order_datetime, Length: 5714, dtype: datetime64[us]
eligible_customers :  Index([  1000,   1001,   1002,   1003,   1004,   1006,   1007,   1008,   1009,
         1010,
       ...
       987819, 987917, 987967, 988465, 988923, 988957, 989031, 989420, 989772,
       989977],
      dtype='int64', name='customer_id', length=5203)
전체 고객 수: 5,714명
관측 완료(90일 창이 6/30 이내인) 고객 수: 5,203명
제외된(아직 90일이 안 지난) 고객 수: 511명


## 20. 첫 구매일로부터 90일 이내 거래만 필터링

In [21]:
# net =  순매출건 join product
# first_month : 최초주문월
# apply 개월수 뽑기
# period_n : 가입후 최조주문월 대비 개월수
# monthly 월별 기간별 매출/마진 집계
# cohort_size : 코호트별 인원수
# first_date : 최초주문일자
# eligible_customers : 관측 완료(90일 창이 6/30 이내인) 고객
net90 = net[net["customer_id"].isin(eligible_customers)].copy()
net90 = net90.merge(first_date.rename("first_date"), on="customer_id")

net90["days_since_first"] = (net90["order_datetime"] - net90["first_date"]).dt.days
net90 = net90[net90["days_since_first"] <= 90]

print(f"90일 이내 거래 라인 수: {len(net90):,}건")
net90[["customer_id", "first_date", "order_datetime", "days_since_first", "margin"]].head()


90일 이내 거래 라인 수: 186,172건


,customer_id,first_date,order_datetime,days_since_first,margin
8,4066,2024-02-14 17:49:14,2024-02-14 17:49:14,0,"334,065.00"
9,4066,2024-02-14 17:49:14,2024-02-14 17:49:14,0,"13,785.00"
12,1802,2024-01-01 01:18:45,2024-01-04 21:10:14,3,615.00
13,1802,2024-01-01 01:18:45,2024-01-04 21:10:14,3,"13,530.00"
22,4823,2024-01-01 00:53:12,2024-01-15 12:55:53,14,"2,616.00"


## 21. 고객별 90일 마진 합산 -> 전체 평균

이 값이 바로 '첫 구매 후 90일 동안 고객 1명이 평균적으로 남기는 마진'입니다.

In [22]:
# net =  순매출건 join product
# first_month : 최초주문월
# apply 개월수 뽑기
# period_n : 가입후 최조주문월 대비 개월수
# monthly 월별 기간별 매출/마진 집계
# cohort_size : 코호트별 인원수
# first_date : 최초주문일자
# eligible_customers : 관측 완료(90일 창이 6/30 이내인) 고객
# net90 : 90일 이내 거래 라인만 추출
per_customer_margin_90d = net90.groupby("customer_id")["margin"].sum()
margin_90d = per_customer_margin_90d.mean()

print(f"90일 시점 1인당 평균 마진: {margin_90d:,.0f}원 (관측 완료 고객 {len(eligible_customers):,}명 대상)")
per_customer_margin_90d.head()


90일 시점 1인당 평균 마진: 935,337원 (관측 완료 고객 5,203명 대상)


customer_id
1000     571,943.00
1001     212,078.00
1002     220,096.00
1003   1,782,726.00
1004     668,431.00
Name: margin, dtype: float64

---
## 22. 획득비 상한 제안 (가정 3개 명시)

In [23]:
print(
    "[가정 3개]\n"
    "  1) 마진 회수 기간을 90일로 본다\n"
    "  2) 90일 마진 전액을 획득비 상한으로 허용한다\n"
    "  3) 6개월 관측 데이터이므로 90일 이후의 장기 가치는 고려하지 않은 보수적 추정이다"
)
print(f"\n[획득비 상한 제안] 고객 1명당 획득비는 약 {margin_90d:,.0f}원을 넘지 않는 것을 권고한다.")


[가정 3개]
  1) 마진 회수 기간을 90일로 본다
  2) 90일 마진 전액을 획득비 상한으로 허용한다
  3) 6개월 관측 데이터이므로 90일 이후의 장기 가치는 고려하지 않은 보수적 추정이다

[획득비 상한 제안] 고객 1명당 획득비는 약 935,337원을 넘지 않는 것을 권고한다.
